# UrjaGram — Google Earth Engine Solar Analysis
Pulls real GHI (Global Horizontal Irradiance) and peak sun hours from NASA/ERA5 datasets for Indian states,
and can export updated solar LUT data back to the React app.

In [ ]:
import ee
import pandas as pd
import json

# Authenticate with your Google account (opens browser once)
ee.Authenticate()

# Initialize with your GEE Cloud project
# Get project ID from: https://console.cloud.google.com
PROJECT_ID = 'your-gee-project-id'  # <-- replace this
ee.Initialize(project=PROJECT_ID)
print('Earth Engine initialized ✓')

In [ ]:
# ── Indian state centroids for peak sun hour calculation ─────────────────────
INDIAN_STATES = {
    'andhra_pradesh':  {'lat': 15.91, 'lng': 79.74},
    'arunachal_pradesh': {'lat': 28.22, 'lng': 94.73},
    'assam':           {'lat': 26.14, 'lng': 91.76},
    'bihar':           {'lat': 25.09, 'lng': 85.31},
    'chhattisgarh':    {'lat': 21.27, 'lng': 81.86},
    'goa':             {'lat': 15.30, 'lng': 74.12},
    'gujarat':         {'lat': 22.25, 'lng': 71.19},
    'haryana':         {'lat': 29.06, 'lng': 76.08},
    'himachal_pradesh':{'lat': 31.10, 'lng': 77.17},
    'jharkhand':       {'lat': 23.61, 'lng': 85.27},
    'karnataka':       {'lat': 15.31, 'lng': 75.71},
    'kerala':          {'lat': 10.85, 'lng': 76.27},
    'madhya_pradesh':  {'lat': 22.97, 'lng': 78.65},
    'maharashtra':     {'lat': 19.75, 'lng': 75.71},
    'odisha':          {'lat': 20.94, 'lng': 84.80},
    'punjab':          {'lat': 31.14, 'lng': 75.34},
    'rajasthan':       {'lat': 27.02, 'lng': 74.22},
    'tamil_nadu':      {'lat': 11.12, 'lng': 78.65},
    'telangana':       {'lat': 18.11, 'lng': 79.01},
    'uttar_pradesh':   {'lat': 26.84, 'lng': 80.94},
    'uttarakhand':     {'lat': 30.07, 'lng': 79.09},
    'west_bengal':     {'lat': 22.98, 'lng': 87.85},
}
print(f'{len(INDIAN_STATES)} states loaded')

In [ ]:
# ── Fetch annual GHI from NASA POWER via Earth Engine (ERA5 solar radiation) ──
# Uses ECMWF/ERA5_LAND/HOURLY surface_solar_radiation_downwards
# Averaged over 5 years for stability

def get_peak_sun_hours(lat, lng, year_start=2019, year_end=2023):
    point = ee.Geometry.Point([lng, lat])
    
    # ERA5-Land: surface_solar_radiation_downwards in J/m²
    era5 = ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY') \
        .filterDate(f'{year_start}-01-01', f'{year_end}-12-31') \
        .select('surface_solar_radiation_downwards')
    
    # Sum all hourly values → total J/m² over period
    total = era5.sum()
    
    val = total.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=point,
        scale=11132,  # ERA5 native ~11km resolution
        maxPixels=1
    ).get('surface_solar_radiation_downwards').getInfo()
    
    if val is None:
        return None
    
    # Convert J/m² → kWh/m²/day (÷ 3600 for Wh, ÷ 1000 for kWh, ÷ days)
    n_days = (year_end - year_start + 1) * 365
    ghi_kwh_day = val / 3600 / 1000 / n_days
    
    # Peak sun hours = GHI (kWh/m²/day) assuming 1 kW/m² reference
    return round(ghi_kwh_day, 1)

# Test with Haryana
haryana_psh = get_peak_sun_hours(29.06, 76.08)
print(f'Haryana peak sun hours (ERA5 5yr avg): {haryana_psh}h/day')

In [ ]:
# ── Batch fetch all states (takes ~2-3 minutes) ───────────────────────────────
results = {}
for state, coords in INDIAN_STATES.items():
    psh = get_peak_sun_hours(coords['lat'], coords['lng'])
    results[state] = psh
    print(f'  {state:25s}: {psh}h')

df = pd.DataFrame([
    {'state': state, 'peak_sun_hours': psh}
    for state, psh in results.items()
]).sort_values('peak_sun_hours', ascending=False)

print('\n── Top 5 solar states ──')
print(df.head())

In [ ]:
# ── Export updated solarLUT.js for the React app ──────────────────────────────
lut_lines = ['// Auto-generated by gee_solar_analysis.ipynb — do not edit manually\n',
             '// Peak sun hours derived from ERA5 5-year average (2019–2023)\n\n',
             'export const solarPeakHours = {\n']

for state, psh in sorted(results.items()):
    lut_lines.append(f'  {state}: {psh},\n')

lut_lines.append('};\n')

output_path = 'src/data/solarLUT_gee.js'
with open(output_path, 'w') as f:
    f.writelines(lut_lines)

print(f'Exported to {output_path}')
print('Replace solarLUT.js with solarLUT_gee.js in the React app to use live GEE data.')

In [ ]:
# ── Bonus: fetch monthly GHI for Haryana (for the forecast chart) ─────────────
def get_monthly_ghi(lat, lng, year=2023):
    point = ee.Geometry.Point([lng, lat])
    monthly = []
    for month in range(1, 13):
        start = f'{year}-{month:02d}-01'
        end_month = month + 1 if month < 12 else 1
        end_year = year if month < 12 else year + 1
        end = f'{end_year}-{end_month:02d}-01'
        
        col = ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY') \
            .filterDate(start, end) \
            .select('surface_solar_radiation_downwards') \
            .sum()
        
        val = col.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=point,
            scale=11132,
            maxPixels=1
        ).get('surface_solar_radiation_downwards').getInfo()
        
        if val is not None:
            import calendar
            days = calendar.monthrange(year, month)[1]
            kwh_day = val / 3600 / 1000 / days
            monthly.append({'month': month, 'ghi_kwh_day': round(kwh_day, 2)})
    
    return monthly

haryana_monthly = get_monthly_ghi(29.06, 76.08)
df_monthly = pd.DataFrame(haryana_monthly)
df_monthly['month_name'] = pd.to_datetime(df_monthly['month'], format='%m').dt.strftime('%b')
print(df_monthly[['month_name', 'ghi_kwh_day']].to_string(index=False))
print('\nUse these values to update dashboardForecastData in sampleData.js')